# Lab 10: Monitoring & Observability

## Overview

In this lab you will enable **inference tables** on a Model Serving endpoint, generate synthetic traffic, inspect the captured logs, create a **Lakehouse Monitor** over the inference table, and walk through the feedback loop that closes the MLOps cycle.

| Topic | Detail |
|---|---|
| **Exam Domain** | Evaluation and Monitoring (12%) |
| **Key Skills** | Inference tables · auto_capture_config · Lakehouse Monitor · drift detection · quality metrics · feedback loop |

### What you will do
1. Enable inference tables via `auto_capture_config` to automatically log every request and response.
2. Generate representative traffic by sending 8 test queries to the endpoint.
3. Inspect the inference table in Unity Catalog to verify captured logs.
4. Create a Lakehouse Monitor over the inference table and trigger the first refresh.
5. Review the auto-generated monitoring dashboard and understand which metrics are tracked.
6. Understand the feedback loop: how monitoring alerts feed back into evaluation, improvement, and redeployment.

In [ ]:
%pip install databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG        = "genai_lab"                  # Unity Catalog catalog name
SCHEMA         = "default"                    # Schema inside the catalog
ENDPOINT_NAME  = "genai-lab-agent-endpoint"  # Serving endpoint from Lab 09

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"Endpoint : {ENDPOINT_NAME}")

## A. Enable Inference Tables

**Inference tables** automatically log every request sent to a Model Serving endpoint and the corresponding response into a Delta table in Unity Catalog. No application-level code changes are needed — logging happens transparently at the serving layer.

The `auto_capture_config` object controls:
- **`catalog_name` / `schema_name`** — where in Unity Catalog the table is written.
- **`table_name_prefix`** — prefix for the auto-created table (e.g., `agent_monitoring_payload`).
- **`enabled`** — toggle logging on or off without redeploying the endpoint.

Once enabled, every inference request is durably recorded, enabling auditing, drift detection, and offline evaluation.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import AutoCaptureConfigInput

w = WorkspaceClient()

# Update the endpoint to enable inference table logging
w.serving_endpoints.update_config(
    name=ENDPOINT_NAME,
    auto_capture_config=AutoCaptureConfigInput(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        table_name_prefix="agent_monitoring",
        enabled=True,
    ),
)

# Verify the configuration was applied
endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
acc = endpoint.config.auto_capture_config
print(f"Inference table logging enabled : {acc.enabled}")
print(f"Target catalog                  : {acc.catalog_name}")
print(f"Target schema                   : {acc.schema_name}")
print(f"Table name prefix               : {acc.table_name_prefix}")
print(f"Payload table                   : {acc.catalog_name}.{acc.schema_name}.{acc.table_name_prefix}_payload")

## B. Generate Traffic

Send 8 representative queries to the endpoint. These requests will be captured in the inference table and will serve as the dataset for the Lakehouse Monitor created in section D.

A 2-second pause between requests ensures the traffic is spread across multiple log batches, making the monitoring dashboard more realistic.

In [ ]:
import time

test_queries = [
    "What is retrieval-augmented generation?",
    "How does a vector database work?",
    "Explain the transformer attention mechanism.",
    "What is the difference between fine-tuning and RAG?",
    "How do I reduce hallucinations in an LLM application?",
    "What is Unity Catalog in Databricks?",
    "Describe the LangChain agent executor loop.",
    "What metrics should I track for an LLM serving endpoint?",
]

for i, query in enumerate(test_queries, start=1):
    try:
        response = w.serving_endpoints.query(
            name=ENDPOINT_NAME,
            dataframe_records=[{"input": query}],
        )
        print(f"[{i:02d}] OK  — {query[:60]}")
    except Exception as e:
        print(f"[{i:02d}] Error — {query[:60]} | {e}")
    time.sleep(2)

print(f"\nTraffic generation complete. {len(test_queries)} requests sent.")

## C. Inspect Inference Tables

Inference tables are standard Delta tables in Unity Catalog. Each row represents one request/response pair with full metadata:

| Column | Description |
|---|---|
| `timestamp` | UTC timestamp when the request was received |
| `request` | Full JSON body of the incoming request |
| `response` | Full JSON body of the model's response |
| `execution_time_ms` | End-to-end latency for the serving call |
| `status_code` | HTTP status code (200 = success) |
| `sampling_fraction` | Fraction of traffic captured (1.0 = all requests) |

These tables can be queried directly in SQL, used as inputs to evaluation pipelines, or monitored for drift.

In [ ]:
INFERENCE_TABLE = f"{CATALOG}.{SCHEMA}.agent_monitoring_payload"

# Count rows captured so far
count_df = spark.sql(f"SELECT COUNT(*) AS row_count FROM {INFERENCE_TABLE}")
row_count = count_df.collect()[0]["row_count"]
print(f"Rows captured in inference table: {row_count}")

# Display a sample with the most relevant columns
sample_df = spark.sql(f"""
    SELECT
        timestamp,
        request,
        response,
        execution_time_ms,
        status_code
    FROM {INFERENCE_TABLE}
    ORDER BY timestamp DESC
    LIMIT 10
""")

display(sample_df)

## D. Create a Lakehouse Monitor

A **Lakehouse Monitor** continuously profiles a Delta table and computes statistical metrics over time. Applied to an inference table it tracks:

- **Data quality** — null rates, schema violations, unexpected values.
- **Distribution drift** — how input/output distributions shift compared to a baseline window.
- **Latency trends** — changes in `execution_time_ms` percentiles over time.
- **Volume anomalies** — sudden drops or spikes in request count.

The monitor uses a **time-series analysis type** (`InferenceLog`) for tables that represent predictions, which pairs a request column with a response column and optionally a ground-truth label column for accuracy tracking.

After creation, `run_refresh` triggers the first profile computation so results appear immediately in the dashboard.

In [ ]:
from databricks.sdk.service.catalog import (
    MonitorInferenceLog,
    MonitorInferenceLogProblemType,
)

INFERENCE_TABLE = f"{CATALOG}.{SCHEMA}.agent_monitoring_payload"

try:
    monitor = w.quality_monitors.create(
        table_name=INFERENCE_TABLE,
        inference_log=MonitorInferenceLog(
            granularities=["1 day"],
            timestamp_col="timestamp",
            model_id_col="endpoint_name",
            prediction_col="response",
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_QUESTION_ANSWERING,
        ),
        assets_dir=f"/Shared/monitoring/lab10",
        output_schema_name=f"{CATALOG}.{SCHEMA}",
    )
    print(f"Monitor created: {monitor.monitor_version}")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Monitor already exists — skipping creation.")
    else:
        raise

# Trigger the first refresh so dashboard metrics are populated
refresh = w.quality_monitors.run_refresh(table_name=INFERENCE_TABLE)
print(f"Refresh triggered. Refresh ID: {refresh.refresh_id}")
print(f"Status: {refresh.state}")
print(f"\nView the dashboard at: Catalog > {CATALOG} > {SCHEMA} > agent_monitoring_payload > Quality tab")

## E. Review Monitoring Dashboard

### Where to find the dashboard

After `run_refresh` completes (typically 2–5 minutes), the Lakehouse Monitor dashboard is accessible in two places:

1. **Unity Catalog Explorer** → open `agent_monitoring_payload` → click the **Quality** tab.
2. **Databricks Dashboards** → the monitor automatically creates a dashboard in the `assets_dir` path specified during creation (`/Shared/monitoring/lab10`).

### Metrics tracked

| Category | Metrics |
|---|---|
| **Volume** | Request count per granularity window, percent change vs. previous window |
| **Latency** | p50 / p90 / p99 `execution_time_ms`, trend over time |
| **Data quality** | Null rate per column, schema change detection |
| **Distribution drift** | Jensen-Shannon divergence for categorical and numeric columns vs. baseline |
| **Error rate** | Fraction of requests with non-200 `status_code` |

### Setting alerts

From the dashboard you can attach **Databricks SQL Alerts** to any metric query:

1. Open the monitor's metric table (e.g., `agent_monitoring_payload_profile_metrics`).
2. Write a SQL query — for example: `SELECT p90_execution_time_ms FROM ... WHERE window_start > current_date() - 1`.
3. Save the query as a **Databricks SQL Alert** with a threshold (e.g., alert when p90 latency > 3000 ms).
4. Configure the alert destination: email, Slack webhook, or PagerDuty.

> **Exam tip:** Lakehouse Monitor metrics are stored as ordinary Delta tables (profile metrics and drift metrics). You can query them with SQL, join them to other data, and build custom dashboards on top of them.

## F. Feedback Loop

Monitoring is only valuable if its outputs drive concrete improvements. The complete MLOps feedback loop for a Databricks GenAI application is:

```
Production Endpoint
      │
      ▼
Inference Table  ──►  Lakehouse Monitor  ──►  Dashboard / Alert
                                                      │
                                         Issue detected (drift, quality drop,
                                         latency spike, error rate increase)
                                                      │
                                                      ▼
                                         Re-Evaluate  (Lab 08)
                                         Run LLM-as-a-Judge on new inference
                                         table samples to quantify regression
                                                      │
                                                      ▼
                                         Improve
                                         Update prompts / retrieval / guardrails
                                         Fine-tune or swap foundation model
                                                      │
                                                      ▼
                                         Redeploy  (Lab 09)
                                         Register new model version in UC
                                         Update or A/B test on serving endpoint
                                                      │
                                                      ▼
                                         Validate
                                         Confirm metrics return to baseline
                                         Close the alert
                                                      │
                                                      └──► back to monitoring
```

### Key stages

| Stage | Action | Lab reference |
|---|---|---|
| **Monitor detects issue** | Drift alert or quality metric crosses threshold | This lab (Lab 10) |
| **Re-evaluate** | Run evaluation suite on fresh inference samples | Lab 08 |
| **Improve** | Update RAG pipeline, prompts, or model | Labs 03–07 |
| **Redeploy** | Register new UC model version, update endpoint | Lab 09 |
| **Validate** | Confirm metrics recover; close alert | Lab 10 |

> **Exam tip:** The exam tests whether you understand that monitoring is not a terminal step — it is the trigger for a continuous improvement cycle. Inference tables are the bridge between production behaviour and the evaluation/training pipeline.

## Key Takeaways

| Concept | Summary |
|---|---|
| **Inference Tables** | Delta tables in Unity Catalog automatically populated with every request/response pair; enabled via `auto_capture_config` on the serving endpoint |
| **`auto_capture_config`** | SDK config object specifying catalog, schema, and table prefix for inference logging; can be toggled without redeploying the endpoint |
| **Lakehouse Monitor** | Managed profiling service that computes data quality, distribution, and drift metrics over a Delta table on a configurable schedule |
| **Quality Metrics** | Null rates, schema violations, error rates, and volume tracked per time window; stored as queryable Delta tables |
| **Drift Detection** | Statistical comparison of input/output distributions against a baseline window; surfaces when the model is receiving out-of-distribution inputs |
| **Feedback Loop** | Monitoring → Alert → Re-evaluate (Lab 08) → Improve → Redeploy (Lab 09) → Validate → Monitoring (continuous cycle) |

---

See **[workbook.md](workbook.md)** for the architecture diagram, exam questions, and cost breakdown.